# Phase 4 Memory Comparison

Colab notebook to compare the memory strategies using an already trained classifier and visualize the best memory method.


In [ ]:
!pip install tensorflow pandas numpy scikit-learn networkx tqdm matplotlib seaborn -q

import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

repo_path = Path('/content/AI_Agentic_DL')
branch_name = 'final-pipeline-integration'
if not repo_path.exists():
    !git clone https://github.com/Lawapaul/AI_Agentic_DL.git /content/AI_Agentic_DL

subprocess.run(['git', '-C', str(repo_path), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(repo_path), 'checkout', branch_name], check=True)
subprocess.run(['git', '-C', str(repo_path), 'pull', 'origin', branch_name], check=True)

if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

print('Repo path:', repo_path)
print('Branch:', branch_name)


In [ ]:
import os
from pathlib import Path

base_candidates = [
    Path('/content/drive/MyDrive/Deep Learning Project/AI Agentic'),
    Path('/content/drive/MyDrive/AI Agentic'),
    Path('/content/AI_Agentic_DL'),
]
repo_base = next((path for path in base_candidates if path.exists()), None)
if repo_base is None:
    raise FileNotFoundError(f'No project directory found in: {base_candidates}')

processed_candidates = [
    repo_base / 'data' / 'processed',
    Path('/content/AI_Agentic_DL/data/processed'),
]
processed_path = next((str(path) for path in processed_candidates if path.exists()), None)
if processed_path is None:
    raise FileNotFoundError(f'No processed dataset found in: {processed_candidates}')

model_candidates = [
    repo_base / 'saved_models' / 'hybrid_cnn_lstm_full_2_2m.keras',
    repo_base / 'saved_models' / 'hybrid_cnn_lstm_500k.keras',
    Path('/content/AI_Agentic_DL/saved_models/hybrid_cnn_lstm_full_2_2m.keras'),
    Path('/content/AI_Agentic_DL/saved_models/hybrid_cnn_lstm_500k.keras'),
]
model_path = next((str(path) for path in model_candidates if path.exists()), None)
if model_path is None:
    raise FileNotFoundError(f'No pretrained model found in: {model_candidates}')

output_csv = str(repo_base / 'experiments' / 'results' / 'memory_comparison_results.csv')
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

print('Project base:', repo_base)
print('Processed data:', processed_path)
print('Model path:', model_path)
print('Output CSV:', output_csv)


In [ ]:
!cd /content/AI_Agentic_DL && python experiments/phase4_memory_pipeline.py   --processed_path "{processed_path}"   --model_path "{model_path}"   --compare_all   --batch_size 256   --top_k 5   --output "{output_csv}"


In [ ]:
import pandas as pd

results_df = pd.read_csv(output_csv)
results_df = results_df.sort_values('top1_retrieval_accuracy', ascending=False).reset_index(drop=True)
display(results_df)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

sns.barplot(data=results_df, x='memory_method', y='top1_retrieval_accuracy', palette='viridis', ax=axes[0])
axes[0].set_title('Top-1 Retrieval Accuracy')
axes[0].set_xlabel('Memory Method')
axes[0].set_ylabel('Accuracy')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=results_df, x='memory_method', y='mean_query_time_ms', palette='mako', ax=axes[1])
axes[1].set_title('Mean Query Time')
axes[1].set_xlabel('Memory Method')
axes[1].set_ylabel('Query Time (ms)')
axes[1].tick_params(axis='x', rotation=20)

sns.barplot(data=results_df, x='memory_method', y='memory_coverage', palette='rocket', ax=axes[2])
axes[2].set_title('Memory Coverage')
axes[2].set_xlabel('Memory Method')
axes[2].set_ylabel('Coverage')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

plot_path = os.path.join(os.path.dirname(output_csv), 'memory_comparison_plots.png')
fig.savefig(plot_path, dpi=200, bbox_inches='tight')
print('Saved memory comparison plots:', plot_path)
